### Downloading datasets from kaggle

In [2]:
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset -p data --unzip

Dataset URL: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
License(s): CC-BY-NC-SA-4.0
100%|█████████████████████████████████████| 41.0M/41.0M [00:03<00:00, 13.5MB/s]



### Loading libraries

In [3]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Download NLTK stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/uno/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Loading the datasets and merging them into a single DataFrame

In [4]:

fake = pd.read_csv('./data/Fake.csv')
true = pd.read_csv('./data/True.csv')

fake['label'] = 1
true['label'] = 0

df = pd.concat([fake, true], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

df = df[['text', 'label']]

### Preparing data by cleaning and stemming the text

In [5]:

def prepare_text(text):
    ps = PorterStemmer()
    text = str(text)

    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    words = text.split()
    stop_words = set(stopwords.words('english'))
    words = [ps.stem(word) for word in words if word not in stop_words]
    
    return ' '.join(words)

print("Preparing text... (this may take a few minutes)")
df['text'] = df['text'].apply(prepare_text)
print("Preparing complete!")

Preparing text... (this may take a few minutes)
Preparing complete!


### Cleaning data from unwanted words and phrases

In [ ]:

def clean_text(tekst):
    tekst = str(tekst)

    tekst = re.sub(r"^\s*[A-Za-z .,'/-]{0,60}\(Reuters\)\s*-\s*", "", tekst)

    tekst = re.sub(r"\(reuters\)", " ", tekst, flags=re.IGNORECASE)
    tekst = re.sub(r"\breuters\b", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"(featured image via|image via|featured image|getty images|pic\.twitter\.com|screen capture|screenshot via|photo by)", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"(https?://\S+|www\.\S+|\b\S+\.com\b)", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"@\w+", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"\b(monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"\b(january|february|march|april|may|june|july|august|september|october|november|december)\b", " ", tekst, flags=re.IGNORECASE)

    tekst = re.sub(r"\s+", " ", tekst)
    tekst = tekst.strip()

    return tekst

print("Cleaning text... (this may take a few minutes)")
df["text_ok"] = df["text"].apply(clean_text)
print("Cleaning complete!")
# provera da li je ostalo reci "reuters" u tekstu
df["length"] = df["text_ok"].str.len()
# df = df[df["length"] >= 40]
# df = df.reset_index(drop=True)
koliko = df["text_ok"].str.contains("reuters", case=False).sum()
print("Jos uvek ima rec 'Reuters' u", koliko, "clanaka")
print("Prosecna duzina pre  :", round(df["text"].str.len().mean()))
print("Prosecna duzina posle:", round(df["text_ok"].str.len().mean()))
print()

prave = df[df["label"] == 0]
print("PRE  :", prave["text"].iloc[0][:100])
print("POSLE:", prave["text_ok"].iloc[0][:100])


Cleaning text... (this may take a few minutes)
Cleaning complete!
Jos uvek ima rec 'Reuters' u 18 clanaka
Prosecna duzina pre  : 1535
Prosecna duzina posle: 1520

PRE  : washington reuter u presid donald trump remov chief strategist steve bannon nation secur council wed
POSLE: washington reuter u presid donald trump remov chief strategist steve bannon nation secur council rev


### Checking for empty articles

In [7]:


prazni = df[df["length"] < 40]
puni = df[df["length"] >= 40]

print("praznih clanaka:", len(prazni))
print("  lazne:", len(prazni[prazni["label"] == 1]))
print("  prave:", len(prazni[prazni["label"] == 0]))

df = puni
df = df.reset_index(drop=True)

print()
print("ostalo:", len(df))
print(df["label"].value_counts())

praznih clanaka: 0
  lazne: 0
  prave: 0

ostalo: 44010
label
1    22594
0    21416
Name: count, dtype: int64


# Saving the cleaned DataFrame to a pickle file

In [8]:

df.to_pickle("data/cleaned_data.pkl")